In [92]:
import os
import sys 
os.chdir("/workspaces/dev")
sys.path.append("/workspaces/dev/modules")

In [93]:
from RTWhisper import TokenStreamer, Hyperparameters
import librosa
import numpy as np

In [94]:
MODEL_SIZE = "large-v3"

SAMPLE_RATE = 16000
BUFFER_SIZE = 10

In [95]:
audio, sr = librosa.load("/workspaces/dev/.data/news_with_English.mp3", sr=SAMPLE_RATE)

In [96]:
# audio = audio[85 * SAMPLE_RATE:]

In [97]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=48000, scale=400))
  rand_len = np.clip(rand_len, 46000, 50000)
  end = min(pos + rand_len, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [98]:
# full_text = ""
# for segment in segments:
#   if len(segment) < 160:
#     continue
#   seg, info = whisper.translate(segment, language="ko")
#   for s in seg:
#     full_text += s.text

In [99]:
# print(full_text)

In [100]:
HYPERPARAMETERS = {
  "sentence_max_prev_sentence": 1,
  "weighted_and_offset_token_boundary": 8000,
  "duration_filter_z": {
    "default": 2.0,
    "ko": 2.0,
    "en": 2.0,
  },
  "probability_filter": {
    "z": {
      "default": 2.0,
      "ko": 2.0,
      "en": 2.0,
    },
    "min_prob": {
      "default": 1.0,
      "ko": 0.4,
      "en": 0.4,
    },
  },
  "selector": {
    "search_range_sc": {
      "default": 24000,
      "ko": 24000,
      "en": 24000,
    },
    "threshold": {
      "default": 0.5,
      "ko": 0.25,
      "en": 0.5,
    },
    "padding": {
      "default": 3200,
      "ko": 3200,
      "en": 3200,
    },
    "tolerance": {
      "default": 8000,
      "ko": 8000,
      "en": 8000,
    },
  },
  "classifier_max_prev_sc": {
    "default": 96000
  }
}

In [101]:
hyper = Hyperparameters(None, HYPERPARAMETERS)

In [102]:
whisper_service = TokenStreamer.get_instance(hyper)

In [103]:
raise Exception("stop")

Exception: stop

In [104]:
from RTWhisper.data import Param
from IPython.display import Audio

In [105]:
segment_id = 0
completed = {}
param = Param()

In [ ]:
segment = segments[segment_id]
segment_id += 1

param.audio = segment

result = whisper_service.process(param)
completed.update(result.completed)

print(f"{segment_id}" + "--" * 20)
print([(v.lang, v.text) for k, v in completed.items()])
print([(v.lang, v.text) for v in result.prev_words if v.is_word])
print([(v.lang, v.text) for v in result.prev_recog if v.is_word])

param.update(result)

Audio(result.processed_audio, rate=SAMPLE_RATE)

1----------------------------------------
[]
[]
[('ko', ' bbc')]


In [ ]:
Audio(result.prev_audio, rate=SAMPLE_RATE)

In [106]:
for segment in segments:
  param.audio = segment

  result = whisper_service.process(param)
  completed.update(result.completed)

  # print(f"{segment_id}" + "--" * 20)
  # print([(v.lang, v.text) for k, v in completed.items()])
  # print([(v.lang, v.text) for v in result.prev_words if v.is_word])
  # print([(v.lang, v.text) for v in result.prev_recog if v.is_word])

  param.update(result)

In [107]:
for key, item in completed.items():
  print(key, item)
for v in result.prev_words:
  if v.is_word: print(v.text)
# for v in prev_recog:
#   print(v.text)

0 ['ko'] BBC 생방송 인터뷰도 중에 자녀 난입 사건으로 스타가 된 미국인 교수가 가족이 오늘 카메라 앞에 섰습니다.
1 ['ko'] 스타가 된 4살짜리 딸은 이번엔.
2 ['ko'] 사탕을 입에 물고 등장했습니다.
3 ['ko'] 배영진 기자입니다.
4 ['ko'] 인터뷰 도중 딸과 아들의 등장으로 일약 스타가 된 로버트 켈리 부산대 교수.
5 ['ko'] 유튜브 영상 조회수가,600만 건이 넘는 켈리 교수 가족은 세계적인 유명 인사가 됐습니다.
6 ['ko'] 이렇게 언론의 관심이 커지자 켈리 교수 가족이 기자회견에 나섰습니다.
7 ['ko'] 쳤던 첫째 딸 메리아는 사탕을 물었고 얘기할래?
8 ['ko'] 해놔?
9 ['ko'] 해놨지?
10 ['ko'] 해놔!
11 ['ko'] 해놔!
12 ['ko'] 둘째 아들 존은 엄마 품에 안긴 모습이었습니다. 
13 ['en'] immediately called or texted or emailed the bbc I communicated with the BBC immediately afterwards and I apologized to them. 
14 ['en', 'ko'] I said that if they never called us back or never asked me to be on television 귀여운 춤으로 화제가 된 딸에 대한 질문도 잇따랐습니다. 
15 ['en'] answer that's She's four.
16 ['en'] She has no idea.
17 ['ko'] 당시 BBC와 인터뷰한 내용은 한국의 대통령 탄핵 사건이었습니다. 
18 ['en'] months, six months, millions of people on the streets, no one's car got burned. 
19 ['en'] The protesters even picked up their trash.
20 ['en', 'ko'] I find that that's just a model